In [1]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

In [2]:
df = pd.read_csv("../data/processed/house_prices_refined.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (13297, 5)


,bhk,propertytype,location,sqft,totalprice
0,3,Flat,Ahmedabad,1346,15700000
1,4,Flat,Ahmedabad,1872,17500000
2,4,Flat,Ahmedabad,1650,20200000
3,5,Flat,Ahmedabad,10201,86700000
4,3,Flat,Ahmedabad,968,10400000


In [3]:
feature_columns = [
    "bhk",
    "propertytype",
    "location",
    "sqft"
]

X = df[feature_columns].copy()
y = df["totalprice"].copy()

groups = (
    X.astype(str)
     .agg("||".join, axis=1)
)

print("Total rows:", len(df))
print("Unique feature groups:", groups.nunique())

Total rows: 13297
Unique feature groups: 9359


In [4]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (10556, 4)
X_test shape: (2741, 4)
y_train shape: (10556,)
y_test shape: (2741,)


In [5]:
train_groups = set(groups_train)
test_groups = set(groups_test)

overlap = train_groups.intersection(test_groups)

print("Unique feature groups in train:", len(train_groups))
print("Unique feature groups in test:", len(test_groups))
print("Overlapping feature groups:", len(overlap))

assert len(overlap) == 0, "Group leakage detected!"

Unique feature groups in train: 7487
Unique feature groups in test: 1872
Overlapping feature groups: 0


In [6]:
gss_check = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx_check, test_idx_check = next(
    gss_check.split(X, y, groups=groups)
)

assert train_idx_check.tolist() == train_idx.tolist()
assert test_idx_check.tolist() == test_idx.tolist()

print("Grouped split is reproducible.")

Grouped split is reproducible.


# Grouped Train/Test Split — Conclusion

The dataset contains repeated feature combinations based on:

- `bhk`
- `propertytype`
- `location`
- `sqft`

A standard random train/test split can place the same feature group in both
training and testing data, causing evaluation contamination.

To prevent this, `GroupShuffleSplit` with `random_state=42` is used.

The split was verified to have:

- Zero overlapping feature groups
- Reproducible train/test indices
- Approximately 80/20 row-level separation

This grouped split is used for reliable model evaluation in the project.